In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

GPU 0: Tesla T4 (UUID: GPU-cf5346f8-159a-880b-3d3d-db7cc5aefa34)
Mounted at /content/drive


In [ ]:
!unzip -q /content/drive/MyDrive/yolo_data.zip -d /content/

open('/content/yolo_data/data.yaml', 'w').write("""path: /content/yolo_data
train: images/train
val: images/val
names:
  0: head
""")

!echo "train images:" $(ls /content/yolo_data/images/train | wc -l)
!echo "val images:" $(ls /content/yolo_data/images/val | wc -l)

train images: 1100
val images: 500


In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data="/content/yolo_data/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    project="/content/drive/MyDrive/headcount_runs",
    name="baseline",
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_data/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, fr

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ee097f80050>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
import cv2, glob, os, shutil
from ultralytics import YOLO

src, dst = "/content/yolo_data", "/content/yolo240"
os.makedirs(f"{dst}/images/val", exist_ok=True)
shutil.copytree(f"{src}/labels/val", f"{dst}/labels/val", dirs_exist_ok=True)
for p in glob.glob(f"{src}/images/val/*"):
    im = cv2.imread(p)
    s = 240 / max(im.shape[:2])
    im = cv2.resize(im, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    cv2.imwrite(f"{dst}/images/val/{os.path.basename(p)}", im)

open(f"{dst}/data.yaml", "w").write(f"""path: {dst}
train: /content/yolo_data/images/train
val: images/val
names:
  0: head
""")

m = YOLO("/content/drive/MyDrive/headcount_runs/baseline/weights/best.pt")
for sz in (320, 640):
    r = m.val(data=f"{dst}/data.yaml", imgsz=sz, verbose=False, plots=False)
    print("imgsz", sz, "| mAP50:", round(r.box.map50, 3),
          "| P:", round(r.box.mp, 3), "| R:", round(r.box.mr, 3))

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 576.4±148.8 MB/s, size: 18.2 KB)
val: Scanning /content/yolo240/labels/val... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 1.8Kit/s 0.3s
val: New cache created: /content/yolo240/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 10.5it/s 3.1s
                   all        500      17217      0.569      0.421      0.407      0.129
Speed: 0.2ms preprocess, 1.5ms inference, 0.0ms loss, 1.3ms postprocess per image
imgsz 320 | mAP50: 0.407 | P: 0.569 | R: 0.421
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 737.9±179.7 

In [ ]:
import cv2, glob, os, shutil
from ultralytics import YOLO

src, dst = "/content/yolo_data", "/content/yolo240"
os.makedirs(f"{dst}/images/train", exist_ok=True)
shutil.copytree(f"{src}/labels/train", f"{dst}/labels/train", dirs_exist_ok=True)
for p in glob.glob(f"{src}/images/train/*"):
    im = cv2.imread(p)
    s = 240 / max(im.shape[:2])
    im = cv2.resize(im, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    cv2.imwrite(f"{dst}/images/train/{os.path.basename(p)}", im)

open(f"{dst}/data.yaml", "w").write(f"""path: {dst}
train: images/train
val: images/val
names:
  0: head
""")

m = YOLO("/content/drive/MyDrive/headcount_runs/baseline/weights/best.pt")
_ = m.train(data=f"{dst}/data.yaml", epochs=20, imgsz=640, batch=16,
            project="/content/drive/MyDrive/headcount_runs", name="train240")

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo240/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/headcount_runs/baseline/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train240, 

In [ ]:
import glob, numpy as np
from ultralytics import YOLO

m = YOLO("/content/drive/MyDrive/headcount_runs/train240/weights/best.pt")
imgs = sorted(glob.glob("/content/yolo240/images/val/*"))

true = []
for p in imgs:
    lp = p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    true.append(len([l for l in open(lp).read().splitlines() if l.strip()]))
true = np.array(true)

for conf in (0.25, 0.4, 0.5):
    pred = np.array([len(r.boxes) for r in
                     m.predict(imgs, imgsz=640, conf=conf, stream=True, verbose=False)])
    print(f"conf {conf}: MAE {np.abs(pred - true).mean():.2f} | "
          f"avg true {true.mean():.1f} | avg pred {pred.mean():.1f} | "
          f"bias {np.mean(pred - true):+.2f}")

conf 0.25: MAE 2.17 | avg true 34.4 | avg pred 33.3 | bias -1.16
conf 0.4: MAE 6.21 | avg true 34.4 | avg pred 28.3 | bias -6.16
conf 0.5: MAE 10.42 | avg true 34.4 | avg pred 24.0 | bias -10.42


In [ ]:
for conf in (0.1, 0.15, 0.2, 0.25, 0.3):
    pred = np.array([len(r.boxes) for r in
                     m.predict(imgs, imgsz=640, conf=conf, stream=True, verbose=False)])
    print(f"conf {conf}: MAE {np.abs(pred - true).mean():.2f} | "
          f"avg pred {pred.mean():.1f} | bias {np.mean(pred - true):+.2f}")

conf 0.1: MAE 6.90 | avg pred 41.1 | bias +6.68
conf 0.15: MAE 3.45 | avg pred 37.5 | bias +3.06
conf 0.2: MAE 2.07 | avg pred 35.1 | bias +0.68
conf 0.25: MAE 2.17 | avg pred 33.3 | bias -1.16
conf 0.3: MAE 3.11 | avg pred 31.6 | bias -2.85
